### 0. Environments

In [1]:
!pip install -q sentence-transformers faiss-gpu-cu12 ir_datasets

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0

In [2]:
import ir_datasets
from tqdm import tqdm
from itertools import islice
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import json
import os

In [3]:
# configurations
SAVE_PATH: str = '/kaggle/working/artifacts'
os.makedirs(SAVE_PATH, exist_ok=True)
EMB_PATH = os.path.join(SAVE_PATH, 'embeddings.npy')
METADATA_PATH = os.path.join(SAVE_PATH, 'doc_ids.jsonl')
FAISS_INDEX_PATH = os.path.join(SAVE_PATH, 'faiss.index')

EMBEDDING_MODEL: str = 'BAAI/bge-small-en-v1.5'
BATCH_SIZE: int = 256
DEVICE = ['cuda:0', 'cuda:1']
NORMALIZE_EMB: bool = True

# config for dry test
QUICK_RUN: bool = True
MAX_SAMPLES: int = 100

### 1. Dataset

In [4]:
!cp -r /kaggle/input/datasets/hongkimgip/msmarco-passage/msmarco-passage /root/.ir_datasets/msmarco-passage

In [5]:
def preview_iter(title, iterator, fields, n=5):
    print(f"\n========== {title} ==========")

    for i, item in enumerate(islice(iterator, n), start=1):
        print(f"\n--- {title[:-1].title()} {i} ---")
        for field in fields:
            value = getattr(item, field)
            if field == "text":
                value = value[:500]
            print(f"{field}:", value)


def download_and_preview_msmarco(
    corpus_id="msmarco-passage",
    eval_id="msmarco-passage/dev/small",
    n_samples=1,
):
    passages = ir_datasets.load(corpus_id)
    eval_set = ir_datasets.load(eval_id)

    preview_iter(
        "SAMPLE PASSAGES",
        passages.docs_iter(),
        fields=["doc_id", "text"],
        n=n_samples,
    )

    preview_iter(
        "SAMPLE QUERIES",
        eval_set.queries_iter(),
        fields=["query_id", "text"],
        n=n_samples,
    )

    preview_iter(
        "SAMPLE QRELS",
        eval_set.qrels_iter(),
        fields=["query_id", "doc_id", "relevance"],
        n=n_samples,
    )

    corpus = {}
    queries = {}
    qrels = {}

    print("\n========== LOADING FULL CORPUS ==========")
    for doc in tqdm(passages.docs_iter(), desc="Loading passages"):
        corpus[doc.doc_id] = doc.text

    print(f"Total passages loaded: {len(corpus):,}")

    print("\n========== LOADING QUERIES ==========")
    for query in tqdm(eval_set.queries_iter(), desc="Loading queries"):
        queries[query.query_id] = query.text

    print(f"Total queries loaded: {len(queries):,}")

    print("\n========== LOADING QRELS ==========")
    for qrel in tqdm(eval_set.qrels_iter(), desc="Loading qrels"):
        qrels.setdefault(qrel.query_id, {})[qrel.doc_id] = qrel.relevance

    print(f"Total qrels queries loaded: {len(qrels):,}")

    return corpus, queries, qrels


corpus, _, __ = download_and_preview_msmarco()


========== SAMPLE PASSAGES ==========

--- Sample Passage 1 ---
doc_id: 0
text: The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.

========== SAMPLE QUERIES ==========

--- Sample Querie 1 ---
query_id: 1048585
text: what is paula deen's brother

========== SAMPLE QRELS ==========

--- Sample Qrel 1 ---
query_id: 300674
doc_id: 7067032
relevance: 1

========== LOADING FULL CORPUS ==========


Loading passages: 8841823it [00:35, 249864.60it/s]


Total passages loaded: 8,841,823

========== LOADING QUERIES ==========


Loading queries: 6980it [00:00, 446822.27it/s]


Total queries loaded: 6,980

========== LOADING QRELS ==========


Loading qrels: 7437it [00:00, 423341.05it/s]

Total qrels queries loaded: 6,980


### 2. Create embeddings

In [6]:
# =========================
# 1. Prepare docs
# =========================

model = SentenceTransformer(EMBEDDING_MODEL)

if QUICK_RUN:
    doc_ids = list(corpus.keys())[:MAX_SAMPLES]
else:
    doc_ids = list(corpus.keys())

texts = [corpus[doc_id] for doc_id in doc_ids]


# =========================
# 2. Encode embeddings
# =========================

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    normalize_embeddings=NORMALIZE_EMB,
    show_progress_bar=True,
)

embeddings = embeddings.astype("float32")

assert len(doc_ids) == embeddings.shape[0]


# =========================
# 3. Save embeddings + metadata
# =========================

np.save(EMB_PATH, embeddings)

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    for row_id, doc_id in enumerate(doc_ids):
        f.write(json.dumps({
            "row_id": row_id,
            "doc_id": doc_id,
        }, ensure_ascii=False) + "\n")

print("Saved embeddings:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chunks:   0%|          | 0/20 [00:00<?, ?it/s]

Saved embeddings: (100, 384)


#### Save as FAISS index:

In [7]:
# =========================
# 4. Build FAISS index
# =========================

dim = embeddings.shape[1]

# Nếu embeddings đã normalize thì dùng Inner Product tương đương cosine similarity
if NORMALIZE_EMB:
    base_index = faiss.IndexFlatIP(dim)
else:
    # Nếu chưa normalize, có thể dùng L2
    base_index = faiss.IndexFlatL2(dim)

# Bọc bằng IDMap để FAISS trả về row_id đúng với metadata
index = faiss.IndexIDMap2(base_index)

row_ids = np.arange(len(doc_ids)).astype("int64")

index.add_with_ids(embeddings, row_ids)

assert index.ntotal == len(doc_ids)

faiss.write_index(index, FAISS_INDEX_PATH)

print("Saved FAISS index:", FAISS_INDEX_PATH)

Saved FAISS index: /kaggle/working/artifacts/faiss.index
